# 14y Verify VeRi and CityFlow ReID Checkpoints

This kernel depends on PRs #28 and #29 being merged into master before pushing to Kaggle.

It verifies three single-camera ReID checkpoint contracts from a fresh master clone: Eval A CityFlowV2 TransReID mAP, Eval B CLIP-SENet v6 VeRi-776 cosine and rerank+AQE metrics, and Eval C 09v TransReID VeRi-776 SIE metrics. Each eval records its own PASS/FAIL rows so a failure in one target does not hide the others; the final cell raises only after the full summary is written.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys
import traceback

REPO_URL = "https://github.com/MRKDaGods/gp.git"
EXPECTED_MASTER_SHA_AT_BUILD = "d3f5e0e2c673bf28ba4d193acd646d1cac9aa0fb"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"
RESULTS_PATH = WORK_DIR / "14y_verify_results.json"
EVAL_JSON_DIR = WORK_DIR / "14y_eval_json"
EVAL_JSON_DIR.mkdir(parents=True, exist_ok=True)

results = []
eval_outputs = {}
context = {
    "verifier": "14y_verify_veri_reid_checkpoints",
    "expected_master_sha_at_build": EXPECTED_MASTER_SHA_AT_BUILD,
    "device": "cuda:0",
    "datasets": {},
    "checkpoints": {},
}


def run(cmd, *, cwd=None, check=True, capture=False):
    rendered = " ".join(map(str, cmd))
    print("$", rendered)
    if capture:
        return subprocess.check_output(list(map(str, cmd)), cwd=cwd, text=True)
    return subprocess.run(list(map(str, cmd)), cwd=cwd, check=check, text=True)


def pip_install(*args):
    run([sys.executable, "-m", "pip", "install", "-q", *args])


def clone_or_refresh_master():
    if not PROJECT.exists():
        run(["git", "clone", "--branch", "master", "--depth", "1", REPO_URL, str(PROJECT)])
    else:
        run(["git", "-C", PROJECT, "fetch", "origin", "master", "--depth", "1"])
        run(["git", "-C", PROJECT, "checkout", "master"])
        run(["git", "-C", PROJECT, "reset", "--hard", "origin/master"])
    os.chdir(PROJECT)
    sys.path.insert(0, str(PROJECT))
    head_sha = run(["git", "rev-parse", "HEAD"], capture=True).strip()
    context["git_sha"] = head_sha
    print("resolved git SHA:", head_sha)
    print("expected master SHA at notebook build:", EXPECTED_MASTER_SHA_AT_BUILD)
    if head_sha != EXPECTED_MASTER_SHA_AT_BUILD:
        print(f"WARN: master moved since notebook build: expected {EXPECTED_MASTER_SHA_AT_BUILD}, got {head_sha}")


def record_metric(label, observed, target, tolerance=0.005, required=True, error=None, extra=None):
    if observed is None:
        passed = False if required else True
        delta = None
    else:
        observed = float(observed)
        delta = observed - float(target)
        passed = abs(delta) <= float(tolerance)
    row = {
        "label": label,
        "observed": observed,
        "target": float(target),
        "tolerance": float(tolerance),
        "delta": delta,
        "status": "PASS" if passed else "FAIL",
        "required": bool(required),
    }
    if error:
        row["error"] = str(error)
    if extra:
        row.update(extra)
    results.append(row)
    print(f"{row['status']:4} {label}: observed={observed} target={target} delta={delta}")
    if error:
        print("  error:", error)
    return row


def record_exception(prefix, targets, exc):
    message = "".join(traceback.format_exception_only(type(exc), exc)).strip()
    print(f"{prefix} failed: {message}")
    traceback.print_exc()
    for label, target in targets:
        record_metric(label, None, target, error=message)


def metric_from(data, *path_options):
    for path in path_options:
        cur = data
        ok = True
        for key in path:
            if isinstance(cur, dict) and key in cur:
                cur = cur[key]
            else:
                ok = False
                break
        if ok and cur is not None:
            return float(cur)
    raise KeyError(f"None of the metric paths existed: {path_options}")


clone_or_refresh_master()
print("python:", sys.version)
print("platform:", platform.platform())

pip_install("faiss-cpu", "motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click")
pip_install("filterpy", "ftfy", "lapx", "scikit-learn", "scipy", "pandas", "opencv-python-headless", "tqdm")
pip_install("timm==1.0.11", "open_clip_torch==2.30.0", "pretrainedmodels==0.7.4")
pip_install("--no-deps", "ultralytics")
pip_install("--no-deps", "boxmot==11.0.3")
pip_install("--no-deps", "-e", ".")

try:
    import torch
    print("cuda available:", torch.cuda.is_available())
    assert torch.cuda.is_available(), "14y requires GPU (T4/P100 acceptable)"
except Exception:
    raise

for mount in [Path("/tmp"), WORK_DIR]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount}: {free / 1024**3:.1f} GiB free / {total / 1024**3:.1f} GiB total")

In [ ]:
INPUT_ROOT = Path("/kaggle/input")


def list_input_roots(max_items=100):
    roots = sorted(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
    for item in roots[:max_items]:
        print(item)
    if len(roots) > max_items:
        print(f"... {len(roots) - max_items} more")
    return roots


def find_file(names, hints=()):
    matches = []
    lowered_hints = tuple(h.lower() for h in hints)
    for name in names:
        matches.extend(INPUT_ROOT.rglob(name) if INPUT_ROOT.exists() else [])
    if lowered_hints:
        filtered = [p for p in matches if all(h in str(p).lower() for h in lowered_hints)]
        matches = filtered or matches
    matches = sorted(set(matches), key=lambda p: (len(str(p)), str(p)))
    if not matches:
        raise FileNotFoundError(f"Could not find {names} hints={hints}")
    print("selected", matches[0])
    return matches[0]


def symlink_or_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    if dst.exists() or dst.is_symlink():
        if dst.is_dir() and not dst.is_symlink():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        dst.symlink_to(src, target_is_directory=src.is_dir())
    except OSError:
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)


def find_veri_root():
    known = [
        INPUT_ROOT / "veri-vehicle-re-identification-dataset",
        INPUT_ROOT / "datasets" / "abhyudaya12" / "veri-vehicle-re-identification-dataset",
    ]
    for root in known:
        if (root / "image_query").exists() and (root / "image_test").exists():
            return root
    for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
        if path.is_dir() and (path / "image_query").exists() and (path / "image_test").exists():
            return path
    raise FileNotFoundError("VeRi-776 root with image_query/image_test not found")


def find_cityflow_root():
    known = [
        INPUT_ROOT / "data-aicity-2023-track-2",
        INPUT_ROOT / "datasets" / "thanhnguyenle" / "data-aicity-2023-track-2",
    ]
    for root in known:
        if root.exists():
            return root
    for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
        if path.is_dir() and ((path / "train").exists() or (path / "AIC22_Track1_MTMC_Tracking").exists()):
            return path
    raise FileNotFoundError("CityFlowV2/AIC22 dataset root not found")


list_input_roots()
VERI_ROOT = find_veri_root()
CITYFLOW_ROOT = find_cityflow_root()
context["datasets"] = {"cityflowv2_root": str(CITYFLOW_ROOT), "veri776_root": str(VERI_ROOT)}

Path("models/reid").mkdir(parents=True, exist_ok=True)
CF_CKPT = find_file(["transreid_cityflowv2_best.pth"], hints=("mtmc", "weights", "reid"))
VERI_CKPT = find_file(["vehicle_transreid_vit_base_veri776.pth"], hints=("mtmc", "weights", "reid"))
CLIPSENET_CKPT = find_file(["clipsenet_v6_veri776_best.pth", "vehicle_clip_senet_veri776.pth", "best.pth"], hints=("13", "clip", "senet"))

CF_LOCAL = PROJECT / "models" / "reid" / "transreid_cityflowv2_best.pth"
VERI_LOCAL = PROJECT / "models" / "reid" / "vehicle_transreid_vit_base_veri776.pth"
CLIPSENET_LOCAL = PROJECT / "models" / "reid" / "clipsenet_v6_veri776_best.pth"
symlink_or_copy(CF_CKPT, CF_LOCAL)
symlink_or_copy(VERI_CKPT, VERI_LOCAL)
symlink_or_copy(CLIPSENET_CKPT, CLIPSENET_LOCAL)
context["checkpoints"] = {
    "cityflow_transreid": str(CF_CKPT),
    "clipsenet_v6": str(CLIPSENET_CKPT),
    "veri_transreid_09v": str(VERI_CKPT),
}
print(json.dumps(context, indent=2))

In [ ]:
targets = [("Eval A CityFlowV2 TransReID cosine mAP", 0.8153)]
try:
    output_json = EVAL_JSON_DIR / "eval_a_cityflow_transreid.json"
    cmd = [
        sys.executable, "scripts/eval_cityflowv2_reid.py",
        "--weights", str(CF_LOCAL),
        "--data_root", str(CITYFLOW_ROOT),
        "--crop_dir", str(WORK_DIR / "crops" / "cityflowv2_reid_14y"),
        "--img_size", "256", "256",
        "--batch_size", "128",
        "--device", "cuda:0",
        "--output-json", str(output_json),
        "--no-rerank",
    ]
    subprocess.run(cmd, cwd=PROJECT, check=True)
    data = json.loads(output_json.read_text(encoding="utf-8"))
    eval_outputs["eval_a"] = data
    record_metric("Eval A CityFlowV2 TransReID cosine mAP", metric_from(data, ("mAP",), ("map",)), 0.8153)
    try:
        record_metric("Eval A CityFlowV2 TransReID cosine R1", metric_from(data, ("R1",), ("r1",)), 0.9241, required=False)
    except Exception as exc:
        record_metric("Eval A CityFlowV2 TransReID cosine R1", None, 0.9241, required=False, error=exc)
except Exception as exc:
    record_exception("Eval A", targets, exc)

In [ ]:
targets = [
    ("Eval B CLIP-SENet v6 VeRi cosine mAP", 0.8234),
    ("Eval B CLIP-SENet v6 VeRi cosine R1", 0.9654),
    ("Eval B CLIP-SENet v6 VeRi rerank+AQE mAP", 0.9154),
    ("Eval B CLIP-SENet v6 VeRi rerank+AQE R1", 0.9732),
]
try:
    script = PROJECT / "scripts" / "eval" / "eval_clip_senet_veri776.py"
    if not script.exists():
        raise FileNotFoundError("scripts/eval/eval_clip_senet_veri776.py is missing; merge PR #28 into master before pushing this kernel")
    output_json = EVAL_JSON_DIR / "eval_b_clip_senet_veri776.json"
    cmd = [
        sys.executable, str(script),
        "--checkpoint", str(CLIPSENET_LOCAL),
        "--veri-root", str(VERI_ROOT),
        "--img-size", "320", "320",
        "--batch-size", "64",
        "--device", "cuda",
        "--rerank",
        "--aqe-k", "3",
        "--output-json", str(output_json),
    ]
    subprocess.run(cmd, cwd=PROJECT, check=True)
    data = json.loads(output_json.read_text(encoding="utf-8"))
    eval_outputs["eval_b"] = data
    record_metric("Eval B CLIP-SENet v6 VeRi cosine mAP", metric_from(data, ("mAP",), ("base", "mAP"), ("base", "map")), 0.8234)
    record_metric("Eval B CLIP-SENet v6 VeRi cosine R1", metric_from(data, ("R1",), ("base", "R1"), ("base", "r1")), 0.9654)
    record_metric("Eval B CLIP-SENet v6 VeRi rerank+AQE mAP", metric_from(data, ("rerank_aqe", "mAP"), ("rerank_aqe", "map"), ("best", "by_map", "mAP")), 0.9154)
    record_metric("Eval B CLIP-SENet v6 VeRi rerank+AQE R1", metric_from(data, ("rerank_aqe", "R1"), ("rerank_aqe", "r1"), ("best", "by_map", "R1")), 0.9732)
except Exception as exc:
    record_exception("Eval B", targets, exc)

In [ ]:
targets = [
    ("Eval C 09v TransReID VeRi best R1", 0.9833),
    ("Eval C 09v TransReID VeRi best mAP", 0.8997),
    ("Eval C 09v TransReID VeRi joint R1", 0.9815),
    ("Eval C 09v TransReID VeRi joint mAP", 0.8971),
]
try:
    script = PROJECT / "scripts" / "eval" / "eval_09v_transreid_veri776.py"
    if not script.exists():
        raise FileNotFoundError("scripts/eval/eval_09v_transreid_veri776.py is missing; merge PR #29 into master before pushing this kernel")
    output_json = EVAL_JSON_DIR / "eval_c_09v_transreid_veri776.json"
    cmd = [
        sys.executable, str(script),
        "--checkpoint", str(VERI_LOCAL),
        "--veri-root", str(VERI_ROOT),
        "--batch-size", "64",
        "--device", "cuda",
        "--rerank",
        "--aqe-k", "2",
        "--output-json", str(output_json),
    ]
    subprocess.run(cmd, cwd=PROJECT, check=True)
    data = json.loads(output_json.read_text(encoding="utf-8"))
    eval_outputs["eval_c"] = data
    record_metric("Eval C 09v TransReID VeRi best R1", metric_from(data, ("best_r1", "R1"), ("best_r1", "metrics", "R1")), 0.9833)
    record_metric("Eval C 09v TransReID VeRi best mAP", metric_from(data, ("best_map", "mAP"), ("best_map", "metrics", "mAP")), 0.8997)
    record_metric("Eval C 09v TransReID VeRi joint R1", metric_from(data, ("best", "R1"), ("best", "metrics", "R1")), 0.9815)
    record_metric("Eval C 09v TransReID VeRi joint mAP", metric_from(data, ("best", "mAP"), ("best", "metrics", "mAP")), 0.8971)
    try:
        historical_r5 = metric_from(data, ("historical_r5_guard", "R5"), ("R5",))
        record_metric("Eval C 09v historical R5 guard (informational)", historical_r5, 0.984505, tolerance=0.005, required=False)
    except Exception as exc:
        record_metric("Eval C 09v historical R5 guard (informational)", None, 0.984505, tolerance=0.005, required=False, error=exc)
except Exception as exc:
    record_exception("Eval C", targets, exc)

In [ ]:
summary = {
    **context,
    "metrics": results,
    "eval_outputs": eval_outputs,
    "passed": all(row["status"] == "PASS" for row in results if row.get("required", True)),
}
RESULTS_PATH.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
print(f"wrote {RESULTS_PATH}")

print("\n14y ReID verifier summary")
print("| label | observed | target | delta | status |")
print("|---|---:|---:|---:|---|")
for row in results:
    observed = "NA" if row["observed"] is None else f"{row['observed']:.6f}"
    delta = "NA" if row["delta"] is None else f"{row['delta']:+.6f}"
    print(f"| {row['label']} | {observed} | {row['target']:.6f} | {delta} | {row['status']} |")

failures = [row for row in results if row.get("required", True) and row["status"] != "PASS"]
if failures:
    raise AssertionError("14y required metric failures:\n" + json.dumps(failures, indent=2))
print("All required 14y ReID checkpoint metrics passed.")